In [4]:
import pandas as pd
import re

# Charger les données brutes
df = pd.read_csv("livres_bruts.csv")

# Nettoyage description
df["description"] = df["description"].fillna("").apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", " ", x)
)
df["description"] = df["description"].apply(lambda x: re.sub(r"\s+", " ", x).strip())

# Si description vide → utiliser titre
df["description"] = df.apply(
    lambda row: row["title"] if row["description"] == "" else row["description"], axis=1
)

# Prix → float
df["price"] = df["price"].str.replace("£", "").astype(float)

# Disponibilité → int
df["availability"] = df["availability"].str.extract(r"(\d+)").fillna(0).astype(int)

# Note → numérique
rating_map = {"One":1, "Two":2, "Three":3, "Four":4, "Five":5}
df["rating"] = df["rating"].apply(
    lambda x: next((rating_map[w] for w in rating_map if w in x), 0)
)

df.to_csv("livres_nettoyes.csv", index=False)
print("✅ Données nettoyées prêtes pour PostgreSQL")


✅ Données nettoyées prêtes pour PostgreSQL


C:\Users\Saad\AppData\Local\Temp\ipykernel_13808\1949068939.py:22: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["availability"] = df["availability"].str.extract(r"(\d+)").fillna(0).astype(int)


In [5]:
import psycopg2
from psycopg2.extras import execute_values

# Connexion PostgreSQL (⚠️ adapte les infos à ton setup)
conn = psycopg2.connect(
    dbname="LivresDB",
    user="postgres",
    password="raja2020",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# Création table Livres
cur.execute("""
CREATE TABLE IF NOT EXISTS Livres (
    id SERIAL PRIMARY KEY,
    title TEXT,
    description TEXT,
    price FLOAT,
    availability INT,
    image_url TEXT,
    rating INT
);
""")

# Insertion des données
records = df.to_dict(orient="records")
execute_values(
    cur,
    """
    INSERT INTO Livres (title, description, price, availability, image_url, rating)
    VALUES %s
    """,
    [(r["title"], r["description"], r["price"], r["availability"], r["image_url"], r["rating"]) for r in records]
)

conn.commit()
cur.close()
conn.close()
print("📚 Données insérées dans PostgreSQL")

ModuleNotFoundError: No module named 'psycopg2'